# Functional ROC — significance threshold sweep on chr1

**Question.** The functional F1 summaries score every caller at one operating point: its
default significance threshold. A caller that looks conservative there (high precision, low
recall) may simply sit at another point of the same curve. Sweeping the thresholds and
reading the whole curve tells whether one binarization is *better*, rather than only
*differently tuned*.

**Experiment.** Every ENCODE dataset of `config_encode.yaml` outside its outliers —
pooled replicates, everything restricted to **chr1**. Each is scored on the validations it
has the annotation for (see below): RefSeq TSS for all of them, ATAC-seq and RNA-seq where
the dataset has them. A dataset without a sweep on disk is skipped and the Snakemake command
that builds the missing ones is printed. The sweep is the Snakemake target `rules/functional.smk`
builds, its thresholds the `functional_sweep` block of `config_encode.yaml`:

| Binarization | Swept parameter | Default |
|---|---|---|
| OmniPeak | `--fdr` | 0.05 |
| MACS2 | `-q` (broad marks: `-q q --broad-cutoff 2q`) | 0.05 |
| HOMER | `findPeaks -style histone -fdr` | 0.001 |
| ChromHMM | `BinarizeBam -p` (Poisson threshold) | 1e-4 |

Each method is trained jointly across all sweep thresholds — joint 15-state KMeans on the
peak binarizations, joint 15-state ChromHMM `LearnModel` on its binarizations — and
collectively matched to the chr1 part of the dataset's ENCODE reference.

See README.md for more details.

**Scoring.** The functional validations of `utils.FUNCTIONAL_TARGETS` a dataset has the
annotation for: a state family against an independent annotation of the same sample. Over
the chr1 bp:

* TPR (sensitivity, recall) = overlap / annotation bp
* FPR = (family bp − overlap) / (chr1 bp − annotation bp)
* F1 = 2 · overlap / (family bp + annotation bp), as `utils.annotation_f1()`

To maximize collective utility and prevent misallocation of analytical effort:
1. For Direct Experimental Prioritization: Use Average Precision (AP). It eliminates background distortion, heavily penalizes false discoveries within active states, and directly correlates with validation success rates. Average Precision heavily rewards algorithms that span an ultra-conservative, high-precision regime at low recall.
2. For General-Purpose Pipeline Deployment: Use Mean / Default F1. It ensures the algorithm operates near an optimal balance of coverage and purity out of the box without requiring manual re-calibration.
3. For Fundamental Model Separability: Use Zoomed ROC-AUC (restricted to low $\text{FPR} \le 0.05$). It assesses pure signal-to-noise differentiation while discarding irrelevant high-background regions. AUC is dominated by the highest achieved sensitivity, given that, in a genomic setting, FPR is nearly 0 across all cases.


Tables and figures (PDF) are written to `out/functional`.

## Setup

In [1]:
import contextlib
import glob
import gzip
import io
import os
import sys

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = None
    for _p in [os.getcwd()] + sys.path:
        _curr = os.path.abspath(_p) if _p else ""
        while _curr and _curr != os.path.dirname(_curr):
            if os.path.exists(os.path.join(_curr, "scripts/analysis/analyze.py")):
                PROJECT_ROOT = _curr
                break
            _curr = os.path.dirname(_curr)
        if PROJECT_ROOT:
            break
    if not PROJECT_ROOT:
        PROJECT_ROOT = os.getcwd()

with open(os.path.join(PROJECT_ROOT, "config_encode.yaml")) as f:
    config = yaml.safe_load(f)

for _sub in ["scripts/rules", "scripts/analysis"]:
    _dir = os.path.join(PROJECT_ROOT, _sub)
    if _dir not in sys.path:
        sys.path.insert(0, _dir)

import analyze
import utils
from display import show_table
from utils import (CHROMHMM_DEFAULT, F1_DISPLAY, HOMER, MACS2, OMNI,
                   PRECISION_DISPLAY, SENSITIVITY_DISPLAY)

utils.set_style()

workdir = os.path.expanduser(config["workdir"])
os.chdir(workdir)
P = config["params"]
TOOLS = config["tools"]

CHROM = "chr1"
NSTATES = P["n_states"]
# float() first: YAML reads 1e-10 (no dot) as a string. The directory names are
# the thresholds in %g.
THRESHOLDS = [float(t) for t in P["functional_sweep"]["thresholds"]]
PVALUES = [float(t) for t in P["functional_sweep"]["chromhmm_pvalues"]]
DEFAULTS = {OMNI: 0.05, MACS2: 0.05, HOMER: 0.001, CHROMHMM_DEFAULT: 1e-4}

OUT = "out/functional"
os.makedirs(OUT, exist_ok=True)

with open(TOOLS["chromsizes"]) as f:
    CHR_LEN = next(int(line.split()[1]) for line in f if line.split()[0] == CHROM)


def sweep_dir(ds):
    """The sweep of a dataset, as rules/functional.smk lays it out."""
    return f"{ds}/functional_chr1_sweep"


# The datasets: every one outside the config's outliers - RefSeq TSS needs no
# sample data, so each has a validation - with a sweep on disk. Not every sweep
# point exists - at the strictest ChromHMM p a binarization can hold too few
# patterns for the 15 states the information initialization needs - so a sweep
# counts once it has a matched segmentation; the missing points are listed below.
OUTLIERS = set(config.get("outliers") or [])
CANDIDATES = [ds for ds in config["datasets"] if ds not in OUTLIERS]
DATASETS = [ds for ds in CANDIDATES
            if glob.glob(f"{sweep_dir(ds)}/*/*/*_matched.bed")
            + glob.glob(f"{sweep_dir(ds)}/chromhmm_default/*/model/*_matched.bed")]
NO_SWEEP = [ds for ds in CANDIDATES if ds not in DATASETS]

print(f"Working dir : {workdir}")
print(f"Datasets    : {DATASETS}")
print(f"No sweep    : {NO_SWEEP}")
if NO_SWEEP:
    print("Build them with:\n"
          f"for ds in {' '.join(NO_SWEEP)}; do\n"
          "  snakemake -p $ds/functional_chr1_sweep/.done \\\n"
          "  --use-conda --cores all --directory $(pwd) \\\n"
          "  --snakefile $REPO/Snakefile \\\n"
          "  --configfile $REPO/config_encode.yaml \\\n"
          "  --resources homer_tagdir=4 --notemp --keep-going \\\n"
          "  --rerun-incomplete --rerun-trigger mtime;\n"
          "done")
print(f"Thresholds  : {THRESHOLDS}")
print(f"ChromHMM p  : {PVALUES}")

Working dir : /Users/Oleg.Shpynov/data/2026_segmentations/encode
Datasets    : ['imr90', 'monocytes', 'monocytes_mint', 'gm12878_mint', 'spleen', 'heart_right_ventricle', 'sigmoid_colon', 'adrenal_gland', 'tibial_nerve', 'heart_left_ventricle', 'substantia_nigra', 'temporal_lobe']
No sweep    : []
Thresholds  : [1e-10, 1e-08, 1e-06, 0.0001, 0.001, 0.01, 0.05, 0.1, 0.2, 0.5]
ChromHMM p  : [1e-10, 1e-08, 1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.05]


In [2]:
def cell_of(ds):
    return config["datasets"][ds]["cell"]


def marks_of(ds):
    return sorted({m["mark"] for m in config["datasets"][ds]["bams"].values()})


def sweep_beds(ds):
    """(method key, threshold, default?, matched BED) of every sweep point of a dataset."""
    root = sweep_dir(ds)
    for p in PVALUES:
        yield (CHROMHMM_DEFAULT, p, p == DEFAULTS[CHROMHMM_DEFAULT],
               f"{root}/chromhmm_default/{p:g}/model/{cell_of(ds)}_{NSTATES}_dense_matched.bed")
    for caller in (OMNI, MACS2, HOMER):
        for q in THRESHOLDS:
            yield (utils.method_key(caller), q, q == DEFAULTS[caller],
                   f"{root}/{caller}/{q:g}/{caller}_kmeans_states_matched.bed")


def peak_beds(ds, caller, q):
    ext = "peak" if caller == OMNI else "bed"
    return {mark: f"{sweep_dir(ds)}/{caller}/{q:g}/{mark}.{ext}" for mark in marks_of(ds)}


for ds in DATASETS:
    beds = list(sweep_beds(ds))
    missing = [bed for *_, bed in beds if not os.path.exists(bed)]
    print(f"{ds}: {len(beds) - len(missing)} of {len(beds)} matched segmentations")
    for bed in missing[:5]:
        print("  missing", bed)

imr90: 38 of 38 matched segmentations
monocytes: 38 of 38 matched segmentations
monocytes_mint: 38 of 38 matched segmentations
gm12878_mint: 38 of 38 matched segmentations
spleen: 38 of 38 matched segmentations
heart_right_ventricle: 38 of 38 matched segmentations
sigmoid_colon: 38 of 38 matched segmentations
adrenal_gland: 38 of 38 matched segmentations
tibial_nerve: 38 of 38 matched segmentations
heart_left_ventricle: 38 of 38 matched segmentations
substantia_nigra: 38 of 38 matched segmentations
temporal_lobe: 38 of 38 matched segmentations


## Functional annotations on chr1

The annotations `utils.FUNCTIONAL_TARGETS` scores, built as the analysis builds them: the
dataset's ATAC-seq peaks (extended by ±500 bp), the RNA-seq expressed TSS ± 2 kb and gene bodies
(`analyze.make_expressed_annotations()`), and the RefSeq TSS ± 2 kb of the ChromHMM
coordinates. A dataset is scored on the validations it has the annotation for: RefSeq TSS
needs no sample data, so it is the one every dataset has. Only the features on chr1 are
kept, as the segmentations are chr1 only; the expressed genes are the top 12,000 by TPM
genome-wide, as everywhere else, of which those on chr1 are kept, so "expressed" means the
same as in the genome-wide analysis. Each is merged, so overlaps count every bp once.

In [3]:
def merge(intervals):
    """Sorted, merged (n, 2) array of [start, end) intervals."""
    arr = np.asarray(sorted((int(s), int(e)) for s, e in intervals if e > s), dtype=np.int64)
    if len(arr) == 0:
        return arr.reshape(0, 2)
    out = [arr[0].tolist()]
    for s, e in arr[1:]:
        if s > out[-1][1]:
            out.append([s, e])
        elif e > out[-1][1]:
            out[-1][1] = e
    return np.asarray(out, dtype=np.int64)


def total_bp(iv):
    return int((iv[:, 1] - iv[:, 0]).sum())


def bed_rows(path):
    """(chrom, start, end) of every interval of a BED file."""
    _open = gzip.open if path.endswith(".gz") else open
    with _open(path, "rt") as f:
        return [(c, int(s), int(e)) for c, s, e, *_ in
                (line.rstrip("\n").split("\t") for line in f if not line.startswith("#"))]


def bed_chr1(path):
    return merge((s, e) for c, s, e in bed_rows(path) if c == CHROM)


REFSEQ_TSS = bed_rows(os.path.join(TOOLS["coords_dir"], "RefSeqTSS2kb.hg38.bed.gz"))


def load_features(ds):
    """(chrom, start, end) of every feature of a dataset, genome-wide, by annotation."""
    cfg = config["datasets"][ds]
    features = {"RefSeqTSS2kb": REFSEQ_TSS}
    if cfg.get("atac"):
        atac_rows = bed_rows(f"{ds}/atac_{cfg['atac']}.bed.gz")
        features[f"atac_{cfg['atac']}"] = utils.extend_bed(atac_rows, utils.ATAC_WINDOW)
    if cfg.get("rnaseq"):
        with contextlib.redirect_stderr(io.StringIO()):
            expressed = analyze.make_expressed_annotations(f"{ds}/rnaseq_{cfg['rnaseq']}.tsv",
                                                           TOOLS["gencode_gtf"])
        for label, rows in expressed:
            if label in ("ExpressedTSS2kb", "ExpressedGeneBodies"):
                features[label] = [(c, int(s), int(e)) for c, s, e, *_ in rows]
    return features


features, annotations, validations = {}, {}, {}
for ds in DATASETS:
    features[ds] = load_features(ds)
    # The chr1 features only, merged.
    annotations[ds] = {name: merge((s, e) for c, s, e in rows if c == CHROM)
                       for name, rows in features[ds].items()}
    for name, iv in annotations[ds].items():
        assert len(iv) and iv[0, 0] >= 0 and iv[-1, 1] <= CHR_LEN + 2000, (ds, name)
    # (key, validation label, state family, annotation) of every validation scored.
    validations[ds] = []
    for key, label, states, prefix in utils.FUNCTIONAL_TARGETS:
        names = [a for a in annotations[ds] if a.startswith(prefix)]
        if names:
            validations[ds].append((key, label, frozenset(states), names[0]))

annotation_table = pd.DataFrame([
    {"Dataset": ds, "Validation": label, "Annotation": ann,
     "Features (genome)": len(features[ds][ann]),
     "Features (chr1)": sum(c == CHROM for c, _, _ in features[ds][ann]),
     "Merged intervals": len(annotations[ds][ann]),
     "Mb": total_bp(annotations[ds][ann]) / 1e6,
     "chr1 share": total_bp(annotations[ds][ann]) / CHR_LEN}
    for ds in DATASETS for _, label, _, ann in validations[ds]])
annotation_table.to_csv(f"{OUT}/annotations.tsv", sep="\t", index=False, float_format="%.4g")
show_table(f"{OUT}/annotations.tsv")

,Dataset,Validation,Annotation,Features (genome),Features (chr1),Merged intervals,Mb,chr1 share
0,imr90,Active open chromatin vs ATAC-seq,atac_ENCFF243NTP,265247,25258,13445,23.610,0.09484
1,imr90,Core Tss states vs expressed TSS,ExpressedTSS2kb,12000,1226,1057,4.373,0.01756
2,imr90,Core Tss states vs RefSeq TSS,RefSeqTSS2kb,36511,3254,2498,10.580,0.04249
3,imr90,Tx states vs expressed gene bodies,ExpressedGeneBodies,12000,1226,1042,67.280,0.27030
4,monocytes,Core Tss states vs expressed TSS,ExpressedTSS2kb,12000,1238,1062,4.368,0.01754
5,monocytes,Core Tss states vs RefSeq TSS,RefSeqTSS2kb,36511,3254,2498,10.580,0.04249
6,monocytes,Tx states vs expressed gene bodies,ExpressedGeneBodies,12000,1238,998,60.090,0.24140
7,monocytes_mint,Core Tss states vs RefSeq TSS,RefSeqTSS2kb,36511,3254,2498,10.580,0.04249
8,gm12878_mint,Core Tss states vs RefSeq TSS,RefSeqTSS2kb,36511,3254,2498,10.580,0.04249
9,spleen,Active open chromatin vs ATAC-seq,atac_ENCFF105FRE,59146,5661,2982,5.165,0.02075


True

## Peak footprint along the sweep

Mb of chr1 each caller calls at every threshold, summed over the marks — what the threshold
actually changes before any segmentation sees it; the per-mark numbers are in
`peak_stats.tsv`. The sweep reads the pooled replicates only (`{dataset}/bams_chr1`, cut
from the pooled `{dataset}/bams`); a chr1 BAM cut short leaves its mark without peaks at
the end of the chromosome, so the peaks of the loosest threshold have to reach its last 5%.
Stricter thresholds can call only a handful of peaks for a broad mark, so they are not
checked.

In [4]:
peak_stats = pd.DataFrame([
    {"Dataset": ds, "Caller": utils.display_name(caller), "Threshold": q, "Mark": mark,
     "Peaks": len(iv := bed_chr1(path)), "Mb": total_bp(iv) / 1e6,
     "Last peak end": int(iv[-1, 1]) if len(iv) else 0}
    for ds in DATASETS for caller in (OMNI, MACS2, HOMER) for q in THRESHOLDS
    for mark, path in peak_beds(ds, caller, q).items() if os.path.exists(path)])
peak_stats.to_csv(f"{OUT}/peak_stats.tsv", sep="\t", index=False)
loosest = peak_stats[peak_stats["Threshold"] == max(THRESHOLDS)]
short = loosest[loosest["Last peak end"] < 0.95 * CHR_LEN]
assert short.empty, f"peaks stop before the end of {CHROM}:\n{short}"
peak_stats.pivot_table(index=["Caller", "Threshold"], columns="Dataset", values="Mb",
                       aggfunc="sum").round(1)

Dataset              adrenal_gland  gm12878_mint  heart_left_ventricle  \
Caller Threshold                                                         
homer  1.000000e-10            3.6          50.2                  37.4   
       1.000000e-08            4.0          54.6                  40.4   
       1.000000e-06            4.7          59.2                  43.8   
       1.000000e-04            5.8          65.4                  47.8   
       1.000000e-03            6.6          69.4                  50.6   
       1.000000e-02            7.4          73.2                  52.7   
       5.000000e-02            8.3          76.5                  54.5   
       1.000000e-01            8.6          78.0                  55.3   
       2.000000e-01            9.2          80.2                  56.3   
       5.000000e-01           10.5          82.4                  57.6   
macs2  1.000000e-10            0.6          16.5                  17.7   
       1.000000e-08            0.8          20.7                  22.4   
       1.000000e-06            1.1          27.7                  29.5   
       1.000000e-04            1.5          38.8                  41.6   
       1.000000e-03            1.7          48.5                  51.6   
       1.000000e-02            2.2          63.9                  66.9   
       5.000000e-02            3.1          84.0                  88.1   
       1.000000e-01            4.4          97.0                 103.8   
       2.000000e-01            8.2         115.3                 128.2   
       5.000000e-01           84.4         163.5                 186.0   
omni   1.000000e-10            2.9          46.0                  57.6   
       1.000000e-08            3.3          48.1                  62.0   
       1.000000e-06            3.9          50.5                  67.4   
       1.000000e-04            5.1          53.6                  74.7   
       1.000000e-03            6.3          55.6                  79.1   
       1.000000e-02            9.5          58.1                  84.3   
       5.000000e-02           16.0          60.2                  87.8   
       1.000000e-01           20.6          61.4                  88.9   
       2.000000e-01           23.4          62.3                  89.4   
       5.000000e-01           33.2          62.6                  89.6   

Dataset              heart_right_ventricle  imr90  monocytes  monocytes_mint  \
Caller Threshold                                                               
homer  1.000000e-10                   46.5   39.6       50.9            57.9   
       1.000000e-08                   50.3   43.6       54.8            62.6   
       1.000000e-06                   54.5   48.7       58.8            67.8   
       1.000000e-04                   59.2   54.1       64.6            73.6   
       1.000000e-03                   62.2   58.5       68.6            78.0   
       1.000000e-02                   64.6   62.8       73.2            82.1   
       5.000000e-02                   66.2   65.6       75.9            85.4   
       1.000000e-01                   67.1   67.3       77.7            86.7   
       2.000000e-01                   67.8   69.3       78.8            88.8   
       5.000000e-01                   68.5   71.4       80.1            91.4   
macs2  1.000000e-10                   21.2    5.0       24.7            17.2   
       1.000000e-08                   27.1    6.3       28.9            22.7   
       1.000000e-06                   36.1    8.7       34.1            29.8   
       1.000000e-04                   51.0   14.0       42.4            44.3   
       1.000000e-03                   62.2   18.7       48.4            54.8   
       1.000000e-02                   83.1   26.8       56.7            73.7   
       5.000000e-02                  105.0   35.5       68.7            98.8   
       1.000000e-01                  121.0   42.8       73.2           113.4   
       2.000000e-01            

## Scoring

Precision and sensitivity as `analysis_encode.ipynb` computes them: the per-state overlaps
of `analyze.compute_enrichment()` against the merged chr1 annotations, pooled over the state
family as `compare_methods.py` pools them — the family picked by `utils.is_active_open`,
`utils.is_promoter_core` or `utils.is_tx`, precision the family's overlap over its bp,
sensitivity the sum of its states' sensitivities, the overlap over the annotation bp. TPR
is that sensitivity, FPR the family's unannotated bp over the unannotated bp of chr1.

In [5]:
# The state family of every validation, by the check analysis_encode.ipynb pools
# it with (compare_methods.py).
FAMILY_CHECK = {"functional_atac": utils.is_active_open,
                "functional_tss": utils.is_promoter_core,
                "functional_refseq_tss": utils.is_promoter_core,
                "functional_tx": utils.is_tx}
# The chr1 features of every annotation, as compute_enrichment() reads them.
annotation_items = {ds: [(ann, [(c, s, e) for c, s, e in features[ds][ann] if c == CHROM])
                         for _, _, _, ann in validations[ds]]
                    for ds in DATASETS}


def score(ds, segs, method, threshold, default=False):
    """One row per validation of a dataset: the family's TPR, FPR, precision and F1."""
    segs = [row for row in segs if row[0] == CHROM]
    state_bp = {}
    for _, s, e, name in segs:
        state_bp[name] = state_bp.get(name, 0) + e - s
    enrichment = analyze.compute_enrichment(segs, annotation_items[ds])
    enrichment = enrichment[~enrichment["state"].str.startswith("POOL:")]
    rows = []
    for key, label, _, ann in validations[ds]:
        group = enrichment[enrichment["label"] == ann]
        family = group[group["state"].map(FAMILY_CHECK[key])]
        fam_bp = sum(state_bp[st] for st in family["state"])
        ov = float((family["coverage"] * family["state"].map(state_bp)).sum())
        sensitivity = float(family["sensitivity"].sum())
        ann_bp = total_bp(annotations[ds][ann])
        rows.append({"Dataset": ds, "Method": method, "Threshold": threshold,
                     "Default": default, "Validation": key, "Validation label": label,
                     "Annotation": ann, "Family Mb": fam_bp / 1e6,
                     "TPR": sensitivity, "FPR": (fam_bp - ov) / (CHR_LEN - ann_bp),
                     SENSITIVITY_DISPLAY: sensitivity,
                     PRECISION_DISPLAY: ov / fam_bp if fam_bp else np.nan,
                     F1_DISPLAY: 2 * ov / (fam_bp + ann_bp)})
    return rows


with contextlib.redirect_stderr(io.StringIO()):
    points = pd.DataFrame([r for ds in DATASETS
                           for method, q, default, bed in sweep_beds(ds) if os.path.exists(bed)
                           for r in score(ds, analyze.load_bed(bed), method, q, default)])
points.to_csv(f"{OUT}/roc_points.tsv", sep="\t", index=False)
print(f"{points[['Dataset', 'Method', 'Threshold']].drop_duplicates().shape[0]} segmentations scored")

456 segmentations scored


## ROC AUC

Per (dataset, method, validation): the AUC of the sweep anchored at (0, 0) and (1, 1), the
average precision (AP), the PR-AUC (<5% FPR) restricted to points within low false positive rates (FPR < 0.05),
the mean F1 over the thresholds of the sweep, the best F1 along the sweep with its threshold, and the operating point at the caller's
default. Below the full table, the mean of each over the datasets.

In [6]:
def roc_curve(fpr, tpr):
    """Sweep points sorted by FPR and anchored at (0, 0) and (1, 1)."""
    order = np.lexsort((np.asarray(tpr), np.asarray(fpr)))
    return (np.r_[0.0, np.asarray(fpr)[order], 1.0],
            np.r_[0.0, np.asarray(tpr)[order], 1.0])


def auc(fpr, tpr):
    """Trapezoid area under the anchored ROC curve."""
    x, y = roc_curve(fpr, tpr)
    return float(np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2))


def average_precision(recall, precision):
    """AP = sum over the points, sorted by recall, of (R_n - R_n-1) * P_n, R_0 = 0.

    sklearn's average_precision_score on the sweep points: each point's precision
    weighted by the recall it adds; nothing past the highest recall reached.
    """
    recall = np.asarray(recall, dtype=float)
    precision = np.nan_to_num(np.asarray(precision, dtype=float))
    order = np.lexsort((-precision, recall))
    return float(np.sum(np.diff(np.r_[0.0, recall[order]]) * precision[order]))


rows = []
for (ds, key, method), d in points.groupby(["Dataset", "Validation", "Method"]):
    best = d.loc[d[F1_DISPLAY].idxmax()]
    default = d[d["Default"]]
    dflt = default.iloc[0] if len(default) else pd.Series(dtype=float)
    d_fpr5 = d[d["FPR"] < 0.05]
    pr_auc_fpr5 = average_precision(d_fpr5["TPR"], d_fpr5[PRECISION_DISPLAY]) if len(d_fpr5) else 0.0
    rows.append({"Dataset": ds, "Validation": key, "Method": utils.display_name(method),
                 "method": method, "Points": len(d), "AUC": auc(d["FPR"], d["TPR"]),
                 "AP": average_precision(d["TPR"], d[PRECISION_DISPLAY]),
                 "PR-AUC (<5% FPR)": pr_auc_fpr5,
                 "Mean F1": d[F1_DISPLAY].mean(),
                 "Best F1": best[F1_DISPLAY], "Best threshold": best["Threshold"],
                 "Default TPR": dflt.get("TPR", np.nan), "Default FPR": dflt.get("FPR", np.nan),
                 "Default F1": dflt.get(F1_DISPLAY, np.nan)})

auc_table = (pd.DataFrame(rows)
             .sort_values(["Dataset", "Validation", "AUC"], ascending=[True, True, False])
             .reset_index(drop=True))
auc_table.drop(columns="method").to_csv(f"{OUT}/roc_auc.tsv", sep="	", index=False,
                                        float_format="%.4g")

auc_mean = (auc_table.groupby(["Validation", "Method"])
            .agg(Datasets=("Dataset", "nunique"), AUC=("AUC", "mean"),
                 AP=("AP", "mean"),
                 **{"PR-AUC (<5% FPR)": ("PR-AUC (<5% FPR)", "mean"),
                    "Mean F1": ("Mean F1", "mean"), "Default F1": ("Default F1", "mean"),
                    "Best F1": ("Best F1", "mean")})
            .reset_index()
            .sort_values(["Validation", "AUC"], ascending=[True, False]))
auc_mean.to_csv(f"{OUT}/roc_auc_mean.tsv", sep="	", index=False, float_format="%.4g")
show_table(f"{OUT}/roc_auc_mean.tsv")

,Validation,Method,Datasets,AUC,AP,PR-AUC (<5% FPR),Mean F1,Default F1,Best F1
0,functional_atac,Homer KMeans,7,0.6876,0.18220,0.18220,0.3720,0.3719,0.3812
1,functional_atac,OmniPeak KMeans,7,0.6849,0.16920,0.17030,0.3524,0.3572,0.3709
2,functional_atac,Default ChromHMM,7,0.6702,0.17260,0.17690,0.3492,0.3643,0.3953
3,functional_atac,MACS2 KMeans,7,0.6429,0.17930,0.19850,0.3232,0.3282,0.3876
4,functional_refseq_tss,Default ChromHMM,12,0.5719,0.11950,0.11950,0.2057,0.2213,0.2442
5,functional_refseq_tss,Homer KMeans,12,0.5683,0.10850,0.10850,0.2227,0.2233,0.2317
6,functional_refseq_tss,MACS2 KMeans,12,0.5624,0.10570,0.10570,0.1668,0.1718,0.2160
7,functional_refseq_tss,OmniPeak KMeans,12,0.5556,0.09746,0.09746,0.1768,0.1726,0.1959
8,functional_tss,Homer KMeans,9,0.6370,0.18060,0.18060,0.3719,0.3724,0.3762
9,functional_tss,Default ChromHMM,9,0.6302,0.18280,0.18280,0.3312,0.3544,0.3768


True

### Plotting — ROC and PR curves, F1 along the sweep

A figure per validation with subplots across datasets, under `out/functional`: the ROC of every method
over the FPR the sweeps reach; the precision–recall curve with the F1 iso-lines
(grey); then the F1 against the threshold. The default threshold is ringed in black, and extreme threshold boundaries are marked with symbols.
KMeans is drawn dashed, ChromHMM solid.

In [7]:
import matplotlib.lines as mlines

METHOD_LIST = [CHROMHMM_DEFAULT] + [utils.method_key(c)
                                   for c in (HOMER, MACS2, OMNI)]


def style(method):
    return dict(color=utils.method_color(method),
                linestyle="--" if "kmeans" in method else "-")


def plot_roc(key, label):
    ds_list = [ds for ds in DATASETS if any(k == key for k, *_ in validations[ds])]
    if not ds_list:
        return
    fig, axes = plt.subplots(1, len(ds_list), figsize=(5.5 * len(ds_list), 5), squeeze=False)
    for i, ds in enumerate(ds_list):
        ax = axes[0, i]
        data = points[(points["Dataset"] == ds) & (points["Validation"] == key)]
        aucs = auc_table[(auc_table["Dataset"] == ds)
                         & (auc_table["Validation"] == key)].set_index("method")
        zoom = data["FPR"].max() * 1.1
        for method in METHOD_LIST:
            d = data[data["Method"] == method]
            if d.empty:
                continue
            x, y = roc_curve(d["FPR"], d["TPR"])
            ax.plot(x, y, marker="o", markersize=3, linewidth=1.5, **style(method),
                    label=f"{utils.display_name(method)} (AUC {aucs.loc[method, 'AUC']:.3f})")
            dflt = d[d["Default"]]
            ax.scatter(dflt["FPR"], dflt["TPR"], s=60, facecolor="none", edgecolor="black",
                       linewidth=1.2, zorder=5)
        ax.plot([0, 1], [0, 1], linestyle=":", color="0.7", linewidth=1)
        ax.set_xlim(0, zoom)
        ax.set_ylim(0, 1)
        ax.set_xlabel("False positive rate")
        ax.set_ylabel("True positive rate (sensitivity)" if i == 0 else "")
        ax.grid(alpha=0.3)
        ax.set_title(cell_of(ds), **utils.TITLE_STYLE)
        ax.legend(loc="lower right", fontsize=utils.LABEL_FONTSIZE - 1)
    fig.suptitle(f"{label} — chr1 ROC", **utils.TITLE_STYLE)
    utils.save_fig(fig, f"{OUT}/roc_{key}.pdf", bar_width=False)


def plot_pr(key, label):
    ds_list = [ds for ds in DATASETS if any(k == key for k, *_ in validations[ds])]
    if not ds_list:
        return
    fig, axes = plt.subplots(1, len(ds_list), figsize=(5.5 * len(ds_list), 5), squeeze=False)
    for i, ds in enumerate(ds_list):
        ax = axes[0, i]
        data = points[(points["Dataset"] == ds) & (points["Validation"] == key)]
        aps = auc_table[(auc_table["Dataset"] == ds)
                        & (auc_table["Validation"] == key)].set_index("method")
        # F1 iso-lines: precision p with 2pr / (p + r) = f, for recall r > f / 2.
        for f in np.arange(0.1, 1.0, 0.1):
            r = np.linspace(f / 2 + 1e-3, 1, 200)
            p = f * r / (2 * r - f)
            keep = p <= 1
            ax.plot(r[keep], p[keep], color="0.85", linewidth=0.8, zorder=0)
            ax.text(1.0, f / (2 - f), f"{F1_DISPLAY} {f:.1f}", fontsize=utils.LABEL_FONTSIZE - 1,
                    color="0.6", ha="right", va="bottom")
        for method in METHOD_LIST:
            d = data[data["Method"] == method].sort_values("Threshold")
            if d.empty:
                continue
            color = utils.method_color(method)
            ax.plot(d["TPR"], d[PRECISION_DISPLAY], marker="o", markersize=3, linewidth=1.5,
                    **style(method),
                    label=f"{utils.display_name(method)} (AP {aps.loc[method, 'AP']:.3f})")
            if len(d) > 0:
                ax.scatter([d["TPR"].iloc[0]], [d[PRECISION_DISPLAY].iloc[0]], s=45, marker="^",
                           facecolor=color, edgecolor="black", linewidth=1.0, zorder=5)
                if len(d) > 1:
                    ax.scatter([d["TPR"].iloc[-1]], [d[PRECISION_DISPLAY].iloc[-1]], s=45, marker="v",
                               facecolor=color, edgecolor="black", linewidth=1.0, zorder=5)
            dflt = d[d["Default"]]
            if not dflt.empty:
                ax.scatter(dflt["TPR"], dflt[PRECISION_DISPLAY], s=50, marker="o", facecolor="none",
                           edgecolor="black", linewidth=1.2, zorder=6)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel(f"Recall ({SENSITIVITY_DISPLAY.lower()})")
        ax.set_ylabel(PRECISION_DISPLAY if i == 0 else "")
        ax.set_title(cell_of(ds), **utils.TITLE_STYLE)
        ax.grid(alpha=0.3)
        method_handles, method_labels = ax.get_legend_handles_labels()
        symbol_handles = [
            mlines.Line2D([], [], color="grey", marker="^", linestyle="None", markeredgecolor="black", markeredgewidth=1.0, markersize=6, label="Strict"),
            mlines.Line2D([], [], color="none", marker="o", linestyle="None", markeredgecolor="black", markeredgewidth=1.2, markersize=6.5, label="Default threshold"),
            mlines.Line2D([], [], color="grey", marker="v", linestyle="None", markeredgecolor="black", markeredgewidth=1.0, markersize=6, label="Loose"),
        ]
        ax.legend(method_handles + symbol_handles,
                  method_labels + ["Strict", "Default threshold", "Loose"],
                  loc="upper right", fontsize=utils.LABEL_FONTSIZE - 1)
    fig.suptitle(f"{label} — chr1 precision–recall", **utils.TITLE_STYLE)
    utils.save_fig(fig, f"{OUT}/pr_{key}.pdf", bar_width=False)


def plot_f1(key, label):
    """F1 against the swept threshold, every method on one log axis: the peak
    callers' q / FDR and ChromHMM's BinarizeBam Poisson p are all significance
    thresholds, so they share it."""
    ds_list = [ds for ds in DATASETS if any(k == key for k, *_ in validations[ds])]
    if not ds_list:
        return
    fig, axes = plt.subplots(1, len(ds_list), figsize=(5.5 * len(ds_list), 4.5), squeeze=False)
    for i, ds in enumerate(ds_list):
        ax = axes[0, i]
        data = points[(points["Dataset"] == ds) & (points["Validation"] == key)]
        for method in METHOD_LIST:
            d = data[data["Method"] == method].sort_values("Threshold")
            if d.empty:
                continue
            color = utils.method_color(method)
            ax.plot(d["Threshold"], d[F1_DISPLAY], marker="o", markersize=3, linewidth=1.5,
                    **style(method), label=utils.display_name(method))
            if len(d) > 0:
                ax.scatter([d["Threshold"].iloc[0]], [d[F1_DISPLAY].iloc[0]], s=45, marker="^",
                           facecolor=color, edgecolor="black", linewidth=1.0, zorder=5)
                if len(d) > 1:
                    ax.scatter([d["Threshold"].iloc[-1]], [d[F1_DISPLAY].iloc[-1]], s=45, marker="v",
                               facecolor=color, edgecolor="black", linewidth=1.0, zorder=5)
            dflt = d[d["Default"]]
            if not dflt.empty:
                ax.scatter(dflt["Threshold"], dflt[F1_DISPLAY], s=50, marker="o", facecolor="none",
                           edgecolor="black", linewidth=1.2, zorder=6)
        ax.set_xscale("log")
        ax.set_xlabel("Threshold: peak caller q / FDR, ChromHMM BinarizeBam Poisson p")
        ax.set_ylabel(F1_DISPLAY if i == 0 else "")
        ax.grid(alpha=0.3)
        ax.set_title(cell_of(ds), **utils.TITLE_STYLE)
        if i == len(ds_list) - 1:
            method_handles, method_labels = ax.get_legend_handles_labels()
            symbol_handles = [
                mlines.Line2D([], [], color="grey", marker="^", linestyle="None", markeredgecolor="black", markeredgewidth=1.0, markersize=6, label="Strict"),
                mlines.Line2D([], [], color="none", marker="o", linestyle="None", markeredgecolor="black", markeredgewidth=1.2, markersize=6.5, label="Default threshold"),
                mlines.Line2D([], [], color="grey", marker="v", linestyle="None", markeredgecolor="black", markeredgewidth=1.0, markersize=6, label="Loose"),
            ]
            ax.legend(method_handles + symbol_handles,
                      method_labels + ["Strict", "Default threshold", "Loose"],
                      loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=utils.LABEL_FONTSIZE)
    fig.suptitle(f"{label} — {F1_DISPLAY} along the sweep", **utils.TITLE_STYLE)
    utils.save_fig(fig, f"{OUT}/f1_{key}.pdf", bar_width=False)


for key, label, _, _ in utils.FUNCTIONAL_TARGETS:
    plot_roc(key, label)
    plot_pr(key, label)
    plot_f1(key, label)

  saved out/functional/roc_functional_atac.pdf
  saved out/functional/pr_functional_atac.pdf
  saved out/functional/f1_functional_atac.pdf
  saved out/functional/roc_functional_tss.pdf
  saved out/functional/pr_functional_tss.pdf
  saved out/functional/f1_functional_tss.pdf
  saved out/functional/roc_functional_refseq_tss.pdf
  saved out/functional/pr_functional_refseq_tss.pdf
  saved out/functional/f1_functional_refseq_tss.pdf
  saved out/functional/roc_functional_tx.pdf
  saved out/functional/pr_functional_tx.pdf
  saved out/functional/f1_functional_tx.pdf


### Plotting — Summary metrics across datasets

A combined figure per evaluation metric (`roc_auc`, `ap`, `pr_auc_fpr5`, `f1_mean`), comparing
all methods across the functional validation targets on the x-axis. The bars show the mean ± SE over
datasets, with individual datasets plotted as points.

In [8]:
AUC_PANELS = [("functional_atac", "ATAC"), ("functional_tss", "Expressed TSS"),
              ("functional_refseq_tss", "RefSeq TSS"), ("functional_tx", "Expressed gene bodies")]
AUC_ORDER = utils.group_methods(METHOD_LIST)

val_labels = dict(AUC_PANELS)
val_order = [label for _, label in AUC_PANELS]
auc_table_plot = auc_table[auc_table["Validation"].isin(val_labels.keys())].copy()
auc_table_plot["Validation_Label"] = auc_table_plot["Validation"].map(val_labels)

for metric, name, title, ylim in [
        ("PR-AUC (<5% FPR)", "pr_auc_fpr5", "PR-AUC (<5% FPR)",
         (0, auc_table["PR-AUC (<5% FPR)"].max() * 1.1)),
        ("AUC", "roc_auc", "ROC AUC", (0.5, auc_table["AUC"].max() * 1.04)),
        ("AP", "ap", "Average precision", (0, auc_table["AP"].max() * 1.1)),
        ("Mean F1", "f1_mean", f"Mean {F1_DISPLAY} over the sweep",
         (0, auc_table["Mean F1"].max() * 1.1))]:
    order = [m for m in AUC_ORDER if m in set(auc_table_plot["method"])]
    utils.bar_plot(auc_table_plot, "Validation_Label", metric, order=val_order,
                   hue="method", hue_order=order,
                   palette={m: utils.method_color(m) for m in order},
                   legend=True, legend_title="Method",
                   figsize=(8, 4.5), rotation=0, ylim=ylim,
                   title=f"{title} across functional analyses", ylabel=metric,
                   points={"size": 4, "alpha": 0.6}, path=f"{OUT}/{name}.pdf")

  saved out/functional/pr_auc_fpr5.pdf
  saved out/functional/roc_auc.pdf
  saved out/functional/ap.pdf
  saved out/functional/f1_mean.pdf


### Plotting — Average precision / sensitivity across cell lines

2-way plots of sensitivity (x-axis) vs precision (y-axis) across parameter sweeps for each functional validation target
(`summary_2way_active_atac.pdf`, `summary_2way_tss_exptss.pdf`, `summary_2way_tss.pdf`, `summary_2way_tx.pdf`).
Lines connect the average points for each sweep parameter with extreme boundary and default threshold points marked by symbols, and cross-dataset variance for each sweep point is reported below.

In [9]:
import matplotlib.lines as mlines


def plot_2way_scatter(data, title, xlabel, ylabel, outpath, order=None):
    """2-way plot: average precision / sensitivity line across the parameter sweep,
    printing cross-dataset variance for each sweep point.
    """
    method_order = order if order is not None else AUC_ORDER
    methods = [m for m in method_order if m in data["Method"].values]
    if not methods:
        print(f"  skipping {outpath}: no data")
        return

    print(f"\n{title} — Sweep points variance across datasets:")
    for m in methods:
        m_data = data[data["Method"] == m]
        th_stats = m_data.groupby("Threshold").agg(
            sens_mean=(SENSITIVITY_DISPLAY, lambda s: (s * 100.0).mean()),
            sens_var=(SENSITIVITY_DISPLAY, lambda s: (s * 100.0).var()),
            prec_mean=(PRECISION_DISPLAY, lambda s: (s * 100.0).mean()),
            prec_var=(PRECISION_DISPLAY, lambda s: (s * 100.0).var()),
        ).reset_index().sort_values("Threshold")

        print(f"  {utils.display_name(m)}:")
        for _, row in th_stats.iterrows():
            th_str = f"{row['Threshold']:.1e}" if row["Threshold"] < 0.01 else f"{row['Threshold']}"
            print(f"    Threshold={th_str:<7} | "
                  f"Sensitivity: mean={row['sens_mean']:.2f}%, var={row['sens_var']:.2f} | "
                  f"Precision: mean={row['prec_mean']:.2f}%, var={row['prec_var']:.2f}")

    fig, ax = plt.subplots(figsize=(6, 5))

    for m in methods:
        color = utils.method_color(m)
        label = utils.display_name(m)
        ls = "--" if "kmeans" in m else "-"

        # Sweep line: mean sensitivity and precision for each sweep parameter (Threshold)
        m_data = data[data["Method"] == m]
        sweep_means = m_data.groupby("Threshold")[[SENSITIVITY_DISPLAY, PRECISION_DISPLAY]].mean().reset_index()
        sweep_means = sweep_means.sort_values(by="Threshold")
        sweep_x = sweep_means[SENSITIVITY_DISPLAY] * 100.0
        sweep_y = sweep_means[PRECISION_DISPLAY] * 100.0

        ax.plot(sweep_x, sweep_y, color=color, linestyle=ls, linewidth=1.5,
                marker="o", markersize=3, alpha=0.8, label=label, zorder=3)

        if len(sweep_means) > 0:
            # Extreme points: Strict (strict) and Loose (loose)
            ax.scatter([sweep_x.iloc[0]], [sweep_y.iloc[0]], s=45, marker="^",
                       facecolor=color, edgecolor="black", linewidth=1.0, zorder=5)
            if len(sweep_means) > 1:
                ax.scatter([sweep_x.iloc[-1]], [sweep_y.iloc[-1]], s=45, marker="v",
                           facecolor=color, edgecolor="black", linewidth=1.0, zorder=5)

        # Plot default operating point (ringed circle in black)
        dflt_data = m_data[m_data["Default"]]
        if not dflt_data.empty:
            dflt_th = dflt_data["Threshold"].iloc[0]
            dflt_row = sweep_means[sweep_means["Threshold"] == dflt_th]
            if not dflt_row.empty:
                dx = dflt_row[SENSITIVITY_DISPLAY].iloc[0] * 100.0
                dy = dflt_row[PRECISION_DISPLAY].iloc[0] * 100.0
                ax.scatter([dx], [dy], s=50, marker="o", facecolor="none", edgecolor="black", linewidth=1.2, zorder=6)

    ax.set_title(title, **utils.TITLE_STYLE)
    ax.set_xlabel(xlabel, fontsize=utils.AXIS_FONTSIZE)
    ax.set_ylabel(ylabel, fontsize=utils.AXIS_FONTSIZE)
    ax.tick_params(labelsize=utils.TICK_FONTSIZE)
    ax.grid(alpha=0.3)

    method_handles, method_labels = ax.get_legend_handles_labels()
    symbol_handles = [
        mlines.Line2D([], [], color="grey", marker="^", linestyle="None", markeredgecolor="black", markeredgewidth=1.0, markersize=6, label="Strict"),
        mlines.Line2D([], [], color="none", marker="o", linestyle="None", markeredgecolor="black", markeredgewidth=1.2, markersize=6.5, label="Default threshold"),
        mlines.Line2D([], [], color="grey", marker="v", linestyle="None", markeredgecolor="black", markeredgewidth=1.0, markersize=6, label="Loose"),
    ]
    ax.legend(method_handles + symbol_handles,
              method_labels + ["Strict", "Default threshold", "Loose"],
              fontsize=utils.LEGEND_FONTSIZE, title_fontsize=utils.LEGEND_TITLE_FONTSIZE,
              bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)

    utils.save_fig(fig, outpath, tight=False)


TWOWAY_PLOTS = [
    ("functional_atac", "Active chromatin validation (ATAC-seq)",
     "ATAC-seq peaks bp overlap (Sensitivity, %)",
     "Active states bp overlap (Precision, %)",
     f"{OUT}/summary_2way_active_atac.pdf"),
    ("functional_tss", "Tss state validation (Expressed TSS)",
     "Expressed TSS bp overlap (Sensitivity, %)",
     "Tss state bp overlap (Precision, %)",
     f"{OUT}/summary_2way_tss_exptss.pdf"),
    ("functional_refseq_tss", "Tss state validation (RefSeq TSS)",
     "RefSeq TSS bp overlap (Sensitivity, %)",
     "Tss state bp overlap (Precision, %)",
     f"{OUT}/summary_2way_tss.pdf"),
    ("functional_tx", "Tx state validation (Expressed Gene Bodies)",
     "Expressed gene bodies bp overlap (Sensitivity, %)",
     "Tx state bp overlap (Precision, %)",
     f"{OUT}/summary_2way_tx.pdf"),
]

for val_key, title, xlabel, ylabel, outpath in TWOWAY_PLOTS:
    d = points[points["Validation"] == val_key]
    plot_2way_scatter(d, title, xlabel, ylabel, outpath, order=AUC_ORDER)



Active chromatin validation (ATAC-seq) — Sweep points variance across datasets:
  Default ChromHMM:
    Threshold=1.0e-10 | Sensitivity: mean=34.15%, var=195.02 | Precision: mean=53.12%, var=641.80
    Threshold=1.0e-08 | Sensitivity: mean=36.35%, var=201.62 | Precision: mean=49.97%, var=597.52
    Threshold=1.0e-06 | Sensitivity: mean=38.94%, var=217.06 | Precision: mean=46.15%, var=545.07
    Threshold=1.0e-05 | Sensitivity: mean=40.25%, var=217.26 | Precision: mean=43.59%, var=501.76
    Threshold=1.0e-04 | Sensitivity: mean=41.31%, var=200.36 | Precision: mean=40.65%, var=454.70
    Threshold=1.0e-03 | Sensitivity: mean=41.69%, var=172.11 | Precision: mean=37.32%, var=393.11
    Threshold=0.01    | Sensitivity: mean=38.04%, var=162.91 | Precision: mean=33.40%, var=309.73
    Threshold=0.05    | Sensitivity: mean=34.25%, var=116.66 | Precision: mean=30.12%, var=234.96
  Homer KMeans:
    Threshold=1.0e-10 | Sensitivity: mean=38.08%, var=395.89 | Precision: mean=55.42%, var=576.50
 